# 01: Raw Logs → Training Records

Download ALL DSAT scraper CSVs from GitHub, pair consecutive bus observations (grouped by bus+route+direction), balance direction 1 via oversampling, and export a clean CSV for XGBoost training.

In [ ]:
!pip install pandas numpy haversine requests

In [ ]:
import os, requests, json, pandas as pd
from haversine import haversine, Unit
from collections import defaultdict

# ── 1. dynamically discover ALL route CSVs from macau-gtfs-data ──
BASE = "https://raw.githubusercontent.com/ChiHin-Lio/macau-gtfs-data/main"
RAW_URL = f"{BASE}/raw_logs"

# Load route-metadata to get all known route names
meta_resp = requests.get(f"{BASE}/route-metadata.json")
route_meta = meta_resp.json()
all_route_names = list(route_meta["routes"].keys())
print(f"Known routes from metadata: {len(all_route_names)}")

# Try loading each route CSV from the raw_logs directory
loaded_routes = []
all_dfs = []
for route in all_route_names:
    url = f"{RAW_URL}/{route}.csv"
    try:
        df = pd.read_csv(url)
        if len(df) > 0:
            df["route"] = route
            all_dfs.append(df)
            loaded_routes.append(route)
    except Exception:
        pass  # CSV not yet available for this route

if not all_dfs:
    raise RuntimeError("No route data found. Run the scraper first!")

raw = pd.concat(all_dfs, ignore_index=True)
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
print(f"Loaded {len(loaded_routes)} / {len(all_route_names)} routes with data")
print(f"Total raw records: {len(raw)}")
print(f"Date range: {raw['timestamp'].min()} → {raw['timestamp'].max()}")
print(f"Direction 0: {len(raw[raw['direction']==0])}  |  Direction 1: {len(raw[raw['direction']==1])}")

In [ ]:
# ── 2. load stop coordinates ──
stops_resp = requests.get(f"{BASE}/bus-stops.json")
stops_data = stops_resp.json()
stop_coords = {s["id"]: (s["coordinates"][1], s["coordinates"][0]) for s in stops_data}
print(f"Loaded {len(stop_coords)} stops")

# Also try DSAT official BUS_POLE for better WGS84 coords
try:
    dsat_stops = pd.read_csv(f"{BASE}/dsat-data/BUS_POLE.csv")
    for _, row in dsat_stops.iterrows():
        alias = str(row["P_ALIAS"]).strip()
        if pd.notna(row["lng"]) and pd.notna(row["lat"]):
            stop_coords[alias] = (row["lat"], row["lng"])
    print(f"After merging DSAT coords: {len(stop_coords)}")
except Exception:
    print("DSAT BUS_POLE not available, using ParkGo stops only")

In [ ]:
# ── 3. generate training records ──
# Group by (busPlate, route, direction) to ensure correct pairing
training_records = []
skipped_no_coords = 0

for (bus, route, direction), group in raw.groupby(["busPlate", "route", "direction"]):
    group = group.sort_values("timestamp")
    prev_row = None
    prev_latlon = None
    for _, row in group.iterrows():
        if prev_row is None:
            prev_row = row
            prev_latlon = stop_coords.get(row["staCode"])
            continue
        # Skip same-station observations (bus hasn't moved)
        if row["staCode"] == prev_row["staCode"]:
            continue
        cur_latlon = stop_coords.get(row["staCode"])
        if prev_latlon is None or cur_latlon is None:
            skipped_no_coords += 1
            prev_row = row
            prev_latlon = cur_latlon
            continue
        dt = (row["timestamp"] - prev_row["timestamp"]).total_seconds()
        dist_km = haversine(prev_latlon, cur_latlon, unit=Unit.KILOMETERS)
        if dist_km > 0 and 10 < dt < 1200:
            training_records.append({
                "route": str(route),
                "direction": int(direction),
                "from_stop": prev_row["staCode"],
                "to_stop": row["staCode"],
                "trip_time_s": dt,
                "distance_km": round(dist_km, 4),
                "hour": prev_row["timestamp"].hour,
                "day_of_week": prev_row["timestamp"].weekday(),  # Mon=0, Sun=6
                "timestamp": prev_row["timestamp"].isoformat(),
            })
        prev_row = row
        prev_latlon = cur_latlon

train_df = pd.DataFrame(training_records)
print(f"Training records: {len(train_df)}")
print(f"Skipped (no coords): {skipped_no_coords}")
print(f"Unique segments: {train_df.groupby(['from_stop','to_stop']).ngroups}")
print(f"Unique routes: {train_df['route'].nunique()}")
print(f"Direction balance before oversample:")
print(train_df['direction'].value_counts().to_dict())

In [ ]:
# ── 4. balance direction 1 → 50:50 via oversampling ──
dir_counts = train_df['direction'].value_counts()
count_0 = dir_counts.get(0, 0)
count_1 = dir_counts.get(1, 0)

if count_1 > 0 and count_0 > count_1:
    oversample_factor = count_0 // count_1
    remainder = count_0 % count_1
    
    dir_1 = train_df[train_df['direction'] == 1].copy()
    # Duplicate dir 1 records to match dir 0 count
    copies = [dir_1] * oversample_factor
    if remainder > 0:
        copies.append(dir_1.sample(n=remainder, random_state=42))
    
    train_df = pd.concat(
        [train_df[train_df['direction'] == 0]] + copies,
        ignore_index=True
    )
    print(f"Oversampled direction 1: {count_1} → {len(train_df[train_df['direction']==1])}")
    print(f"Direction balance after: {train_df['direction'].value_counts().to_dict()}")
else:
    print("Direction 0 ≤ Direction 1, no oversampling needed")

total = len(train_df)
print(f"Total records after balancing: {total}")

In [ ]:
# ── 5. feature engineering ──
HOLIDAYS_2026 = [
    "2026-01-01", "2026-01-29", "2026-01-30", "2026-01-31",
    "2026-04-04", "2026-04-05", "2026-05-01", "2026-05-06",
    "2026-06-19", "2026-09-22", "2026-10-01", "2026-10-02",
    "2026-10-29", "2026-12-08", "2026-12-20", "2026-12-21",
    "2026-12-24", "2026-12-25",
]
holiday_set = set(HOLIDAYS_2026)

def assign_zone(stop_code):
    code = str(stop_code)
    prefix = code.split("/")[0]
    if prefix in ["M1","M2","M3"]: return "old_town"
    elif prefix in ["M4","M6"]: return "bridge"
    elif prefix in ["M7","M8"]: return "cotai"
    elif code.startswith("T"): return "taipa"
    return "other"

train_df["is_holiday"] = train_df["timestamp"].apply(
    lambda t: t[:10] in holiday_set
).astype(int)
train_df["zone_type"] = train_df["from_stop"].apply(assign_zone)

print(f"is_holiday balance: {train_df['is_holiday'].value_counts().to_dict()}")
print(f"day_of_week values: {sorted(train_df['day_of_week'].unique())}")
print(f"Zone distribution: {train_df['zone_type'].value_counts().to_dict()}")

In [ ]:
# ── 6. export clean CSV ──
output_cols = [
    "route","direction","from_stop","to_stop",
    "trip_time_s","distance_km","hour","day_of_week","is_holiday","zone_type"
]
train_df[output_cols].to_csv("training_data.csv", index=False)
print(f"Exported {len(train_df)} records to training_data.csv")
print("\nColumns:", output_cols)
train_df.head()

In [ ]:
# ── 7. quick EDA ──
import matplotlib.pyplot as plt

print("=== Trip time distribution ===")
print(train_df["trip_time_s"].describe())

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
train_df["trip_time_s"].hist(bins=50, ax=axes[0])
axes[0].set_title("Trip time distribution (s)")
train_df.groupby("hour")["trip_time_s"].mean().plot(ax=axes[1])
axes[1].set_title("Mean trip time by hour")
train_df.groupby("zone_type")["trip_time_s"].mean().plot(kind="bar", ax=axes[2])
axes[2].set_title("Mean trip time by zone")
train_df.groupby("day_of_week")["trip_time_s"].mean().plot(kind="bar", ax=axes[3])
axes[3].set_title("Mean trip time by day")
axes[3].set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
plt.tight_layout()
plt.show()